In [132]:
import pandas as pd
import re
import difflib

In [107]:
cps = pd.read_csv(r'cellphones_full.csv')
cps.info()
df = cps.copy()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

In [ ]:
antutu = pd.read_csv(r'antutu_score_socket.csv')
att = antutu.copy()

In [109]:
df = df.sort_values(by="Tên").reset_index(drop=True)

In [110]:
def extract_refresh_rate(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)\s*hz', s)
    return float(match.group(1)) if match else 0


In [111]:
df["Tần số quét"] = df["Tính năng màn hình"].apply(extract_refresh_rate)

In [112]:
df.head()

,Tên,Giá,Link,Kích thước màn hình,Công nghệ màn hình,Camera sau,Camera trước,Chipset,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Hệ điều hành,Độ phân giải màn hình,Tính năng màn hình,Loại CPU,Dung lượng RAM,Pin,Tương thích,Cảm biến,Tần số quét
0,apple iphone 5s,Giá Liên Hệ,https://cellphones.com.vn/iphone-5s-16-gb.html,4.0 inches,NaN,"8 MP (f/2.2, 29mm, 1/3"", 1.5 µm), tự động lấy ...","1.2 MP (f/2.4, 31mm), 720p@30fps, nhận diện kh...",Apple A7 APL0698,NaN,16 GB,Nano-SIM,11,640 x 1136 pixels,NaN,2x 1.3 GHz Cyclone (nền tảng ARM v8),NaN,Li-Po 1560 mAh,NaN,NaN,0.0
1,asus rog 6,14.490.000đ,https://cellphones.com.vn/asus-rog-phone-6-12g...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,Qualcomm ® Snapdragon ® 8+ thế hệ 1,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
2,asus rog 6 mediatek,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-6-med...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,MediaTek Dimensity 7000 Series,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
3,asus rog 7 pro,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-7-pro...,NaN,NaN,NaN,NaN,Snapdragon 8 Gen 2,Có,128 GB,NaN,NaN,NaN,NaN,NaN,8 GB,NaN,NaN,NaN,0.0
4,asus rog 8,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-8.html,6.7 inches,AMOLED,"Camera góc rộng: 64 MP, f/1.8, 1/1.72"", 0.8 µm...",NaN,Snapdragon 888 5G,Có,256 GB,NaN,NaN,1080 x 2400 pixels (FullHD+),Tần số quét 165 Hz,NaN,12 GB,6.000 mAh,NaN,NaN,165.0


CLEAN NAME

In [ ]:
def clean_phone_name(raw_name):
    if pd.isna(raw_name):
        return ""

    name = str(raw_name).lower().strip()
    if not name:
        return ""

    # Chuẩn hóa dấu nối và loại bỏ phân đoạn không cần thiết
    name = re.sub(r"[\u2010\u2013\u2014\u2212]", "-", name)
    name = re.sub(r"\s*\|\s*.*$", "", name)
    name = re.sub(r"\bđiện thoại\b", "", name, flags=re.I)
    name = re.sub(r"\b(?:ram|rom)\b", "", name, flags=re.I)

    # Xóa các cụm từ quảng cáo / danh mục không phải model
    patterns_to_delete = [
        r"chính hãng",
        r"vn/?a",
        r"bản quốc tế",
        r"bản chính hãng",
        r"xách tay",
        r"nhập khẩu",
        r"full ?box",
        r"open ?box",
        r"like ?new",
        r"second ?hand",
        r"trả góp",
        r"giá tốt",
        r"giá rẻ",
        r"hàng chính hãng",
        r"hàng.*",
        r"Exynos",
        r"Snapdragon",
        r"special edition",
        r"edition",
        r'china',
        '2021',
        '2022',
        '2023',
        '2024',
        '2025',
        
    ]
    name = re.sub(r"\b(?:" + "|".join(patterns_to_delete) + r")\b", "", name, flags=re.I)

    # Xóa các thông tin mạng và kết nối không phải tên model
    name = re.sub(r"\b(?:4g|5g|nfc|lte|wifi|bluetooth)\b", "", name, flags=re.I)

    # Xóa dung lượng RAM/ROM/ổ cứng
    name = re.sub(r"\b\d+(?:[\.,]\d+)?\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[x×]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[+\/]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)

    # Loại bỏ ký tự không cần và chuẩn hóa khoảng trắng
    name = re.sub(r"[\[\]\(\)\{\}]", " ", name)
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = re.sub(r"\s{2,}", " ", name)
    name = re.sub(r"\b-\b", " ", name)
    name = name.strip()

    return name

In [116]:
df["Tên"] = df["Tên"].apply(clean_phone_name)

In [117]:
df.head()

,Tên,Giá,Link,Kích thước màn hình,Công nghệ màn hình,Camera sau,Camera trước,Chipset,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Hệ điều hành,Độ phân giải màn hình,Tính năng màn hình,Loại CPU,Dung lượng RAM,Pin,Tương thích,Cảm biến,Tần số quét
0,apple iphone 5s,Giá Liên Hệ,https://cellphones.com.vn/iphone-5s-16-gb.html,4.0 inches,NaN,"8 MP (f/2.2, 29mm, 1/3"", 1.5 µm), tự động lấy ...","1.2 MP (f/2.4, 31mm), 720p@30fps, nhận diện kh...",Apple A7 APL0698,NaN,16 GB,Nano-SIM,11,640 x 1136 pixels,NaN,2x 1.3 GHz Cyclone (nền tảng ARM v8),NaN,Li-Po 1560 mAh,NaN,NaN,0.0
1,asus rog 6,14.490.000đ,https://cellphones.com.vn/asus-rog-phone-6-12g...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,Qualcomm ® Snapdragon ® 8+ thế hệ 1,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
2,asus rog 6 mediatek,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-6-med...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,MediaTek Dimensity 7000 Series,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
3,asus rog 7 pro,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-7-pro...,NaN,NaN,NaN,NaN,Snapdragon 8 Gen 2,Có,128 GB,NaN,NaN,NaN,NaN,NaN,8 GB,NaN,NaN,NaN,0.0
4,asus rog 8,Giá Liên Hệ,https://cellphones.com.vn/asus-rog-phone-8.html,6.7 inches,AMOLED,"Camera góc rộng: 64 MP, f/1.8, 1/1.72"", 0.8 µm...",NaN,Snapdragon 888 5G,Có,256 GB,NaN,NaN,1080 x 2400 pixels (FullHD+),Tần số quét 165 Hz,NaN,12 GB,6.000 mAh,NaN,NaN,165.0


In [130]:
def count_matching_names(df1, col1, df2, col2):
    set1 = set(df1[col1].unique())
    set2 = set(df2[col2].unique())
    
    matching = set1 & set2
    
    return {
        'matches': len(matching),
        'df1_unique': len(set1),
        'df2_unique': len(set2),
        'match_rate_df1': f"{(len(matching) / len(set1) * 100):.1f}%",
        'match_rate_df2': f"{(len(matching) / len(set2) * 100):.1f}%",
        'matching_names': sorted(list(matching))
    }

In [120]:
def clean_price(raw_price):
    if pd.isna(raw_price):
        return 0
    s = str(raw_price).lower().strip()
    if re.search(r'liên hệ', s):
        return 0
    
    s = re.sub(r'[đ]', '', s)
    s = s.replace('.', '')

    match = re.search(r'(\d+)', s)
    return int(match.group(1)) if match else 0


In [121]:
df["Giá"] = df["Giá"].apply(clean_price)

In [122]:
df.head()

,Tên,Giá,Link,Kích thước màn hình,Công nghệ màn hình,Camera sau,Camera trước,Chipset,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Hệ điều hành,Độ phân giải màn hình,Tính năng màn hình,Loại CPU,Dung lượng RAM,Pin,Tương thích,Cảm biến,Tần số quét
0,apple iphone 5s,0,https://cellphones.com.vn/iphone-5s-16-gb.html,4.0 inches,NaN,"8 MP (f/2.2, 29mm, 1/3"", 1.5 µm), tự động lấy ...","1.2 MP (f/2.4, 31mm), 720p@30fps, nhận diện kh...",Apple A7 APL0698,NaN,16 GB,Nano-SIM,11,640 x 1136 pixels,NaN,2x 1.3 GHz Cyclone (nền tảng ARM v8),NaN,Li-Po 1560 mAh,NaN,NaN,0.0
1,asus rog 6,14490000,https://cellphones.com.vn/asus-rog-phone-6-12g...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,Qualcomm ® Snapdragon ® 8+ thế hệ 1,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
2,asus rog 6 mediatek,0,https://cellphones.com.vn/asus-rog-phone-6-med...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,MediaTek Dimensity 7000 Series,Có,256 GB,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12 GB,6000 mAh,NaN,NaN,165.0
3,asus rog 7 pro,0,https://cellphones.com.vn/asus-rog-phone-7-pro...,NaN,NaN,NaN,NaN,Snapdragon 8 Gen 2,Có,128 GB,NaN,NaN,NaN,NaN,NaN,8 GB,NaN,NaN,NaN,0.0
4,asus rog 8,0,https://cellphones.com.vn/asus-rog-phone-8.html,6.7 inches,AMOLED,"Camera góc rộng: 64 MP, f/1.8, 1/1.72"", 0.8 µm...",NaN,Snapdragon 888 5G,Có,256 GB,NaN,NaN,1080 x 2400 pixels (FullHD+),Tần số quét 165 Hz,NaN,12 GB,6.000 mAh,NaN,NaN,165.0


In [123]:
def clean_storage(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    if not match:
        return 0
    
    value = float(match.group(1))
    
    if 'tb' in s:
        value = value * 1024
    if 'mb' in s:
        value = value / 1024
    
    return value


In [124]:
df["Dung lượng RAM"] = df["Dung lượng RAM"].apply(clean_storage)
df["Bộ nhớ trong"] = df["Bộ nhớ trong"].apply(clean_storage)

In [125]:
df.head()

,Tên,Giá,Link,Kích thước màn hình,Công nghệ màn hình,Camera sau,Camera trước,Chipset,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Hệ điều hành,Độ phân giải màn hình,Tính năng màn hình,Loại CPU,Dung lượng RAM,Pin,Tương thích,Cảm biến,Tần số quét
0,apple iphone 5s,0,https://cellphones.com.vn/iphone-5s-16-gb.html,4.0 inches,NaN,"8 MP (f/2.2, 29mm, 1/3"", 1.5 µm), tự động lấy ...","1.2 MP (f/2.4, 31mm), 720p@30fps, nhận diện kh...",Apple A7 APL0698,NaN,16.0,Nano-SIM,11,640 x 1136 pixels,NaN,2x 1.3 GHz Cyclone (nền tảng ARM v8),0.0,Li-Po 1560 mAh,NaN,NaN,0.0
1,asus rog 6,14490000,https://cellphones.com.vn/asus-rog-phone-6-12g...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,Qualcomm ® Snapdragon ® 8+ thế hệ 1,Có,256.0,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12.0,6000 mAh,NaN,NaN,165.0
2,asus rog 6 mediatek,0,https://cellphones.com.vn/asus-rog-phone-6-med...,6.78 inches,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,MediaTek Dimensity 7000 Series,Có,256.0,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12.0,6000 mAh,NaN,NaN,165.0
3,asus rog 7 pro,0,https://cellphones.com.vn/asus-rog-phone-7-pro...,NaN,NaN,NaN,NaN,Snapdragon 8 Gen 2,Có,128.0,NaN,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,0.0
4,asus rog 8,0,https://cellphones.com.vn/asus-rog-phone-8.html,6.7 inches,AMOLED,"Camera góc rộng: 64 MP, f/1.8, 1/1.72"", 0.8 µm...",NaN,Snapdragon 888 5G,Có,256.0,NaN,NaN,1080 x 2400 pixels (FullHD+),Tần số quét 165 Hz,NaN,12.0,6.000 mAh,NaN,NaN,165.0


In [126]:
def clean_metrics(raw_value):
    if pd.isna(raw_value):
        return 0.0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    return float(match.group(1)) if match else 0.0

In [127]:
cols_to_clean = ["Kích thước màn hình","Pin"]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_metrics)

In [128]:
df.head()

,Tên,Giá,Link,Kích thước màn hình,Công nghệ màn hình,Camera sau,Camera trước,Chipset,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Hệ điều hành,Độ phân giải màn hình,Tính năng màn hình,Loại CPU,Dung lượng RAM,Pin,Tương thích,Cảm biến,Tần số quét
0,apple iphone 5s,0,https://cellphones.com.vn/iphone-5s-16-gb.html,4.00,NaN,"8 MP (f/2.2, 29mm, 1/3"", 1.5 µm), tự động lấy ...","1.2 MP (f/2.4, 31mm), 720p@30fps, nhận diện kh...",Apple A7 APL0698,NaN,16.0,Nano-SIM,11,640 x 1136 pixels,NaN,2x 1.3 GHz Cyclone (nền tảng ARM v8),0.0,1560.0,NaN,NaN,0.0
1,asus rog 6,14490000,https://cellphones.com.vn/asus-rog-phone-6-12g...,6.78,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,Qualcomm ® Snapdragon ® 8+ thế hệ 1,Có,256.0,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12.0,6000.0,NaN,NaN,165.0
2,asus rog 6 mediatek,0,https://cellphones.com.vn/asus-rog-phone-6-med...,6.78,AMOLED,"Camera chính: 50 MP, f/1.9Camera góc siêu rộn...",12 MP,MediaTek Dimensity 7000 Series,Có,256.0,2 SIM (Nano-SIM),Android 12,1080 x 2448 pixels (FullHD+),"Corning ® Gorilla ® Glass Victus, 165Hz, Độ sá...",NaN,12.0,6000.0,NaN,NaN,165.0
3,asus rog 7 pro,0,https://cellphones.com.vn/asus-rog-phone-7-pro...,0.00,NaN,NaN,NaN,Snapdragon 8 Gen 2,Có,128.0,NaN,NaN,NaN,NaN,NaN,8.0,0.0,NaN,NaN,0.0
4,asus rog 8,0,https://cellphones.com.vn/asus-rog-phone-8.html,6.70,AMOLED,"Camera góc rộng: 64 MP, f/1.8, 1/1.72"", 0.8 µm...",NaN,Snapdragon 888 5G,Có,256.0,NaN,NaN,1080 x 2400 pixels (FullHD+),Tần số quét 165 Hz,NaN,12.0,6.0,NaN,NaN,165.0
